> **Research record.** Cleaned copy of `800-research-users-locations/3300-third-places-neighborhood-retention.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Whether neighbourhoods that gain third places retain more remote workers (matching and difference-in-differences). Helper functions are in `third_places_neighborhood_retention.py`.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# 3300 — Third-place activation and neighbourhood retention

**Question.** Among Gush Dan home statistical areas (2022) that were similar on
**2019 SES cluster (`eshkol_madad2019`)**, **2019 working-age share (`age20_59_pcnt`)**,
**CBD distance**, **amenity weight**, and **January 2020 outbound weekday mobility**,
did areas with **detected third-place activation** shift more weekday time from
**out-of-neighbourhood dwells** to **local named POIs** by the post window?

| Decision | Choice |
|---|---|
| Unit | `yishuv_stat_2022` (home SA of workers with known work location) |
| Pre | January 2020 |
| Post | January and February 2023 (matched on February) |
| Placebo / lead | Calendar 2019, pooled (not January 2019 alone) |
| Treatment | `TREAT_FLAG` from `treat_rule` (default: `has_third_place` if `n_detected >= 1`; counts inside the 500 m centroid buffer only) |
| Match | PSM 1:1 NN on confounders only (eshkol, age, CBD, amenity weight, 2020 outbound and local shares). Third-place counts stay the treated–control contrast. Split-sample DiD: rematch inside 2023 high- vs low-WFH SAs |
| Outcomes | duration shares, retention ratio, hours/worker-day, visits, extensive margin |
| Outbound rule | named POI with `dist_to_home >= 500` m **and** `dist_to_work > 500` m (not `flag_home`; office-adjacent POIs are not outbound) |
| Time of day | duration allocated by overlap to local windows: morning / midday / break / evening (+ all24, daytime) |
| Estimation | two-way FE difference-in-differences (regression), not interval comparison |

**Interpretation rule.** If outbound falls but only `share_home` rises → WFH.
Third-place claim needs **local named-POI share up** (and outbound down) *relative to matched controls*.
Do not use `share_local_tp` (controls are structurally ~0), `share_local_place`
(duplicates all-POI shares), or `share_local_cand_nowork` (composition inside the POI pie, not local vs outbound).

**Mechanism split.** A midday effect means the third place substitutes for the office; an
evening effect means it is a consumption amenity; both means general neighbourhood re-centering.

## Structure

Each part states its **goal** and **steps**. Pipeline:

0. Upload 2019 demographics + SES to BigQuery  
1. Crosswalk 2011 → 2022; SA covariates  
2. Treatment (PU flag + intensity)  
3. Weekday outcomes by time-of-day window  
4. PSM + balance checks  
5. TWFE DiD, event study, triple-diff, pair-FE ANCOVA  
5b. Odds of staying local when out of home  
6. Descriptive composition  
7. Robustness (weights, unmatched, IPW, dose, placebo, radius)


In [ ]:
import os
import sys
import warnings
from pathlib import Path

_proj_dir = os.path.join(sys.prefix, "share", "proj")
os.environ.setdefault("PROJ_DATA", _proj_dir)
os.environ.setdefault("PROJ_LIB", _proj_dir)
try:
    from pyproj import datadir as _pyproj_datadir
    _pyproj_datadir.set_data_dir(_proj_dir)
except Exception:
    pass

project_root = Path("../../..")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
_nb_dir = Path.cwd()
if str(_nb_dir) not in sys.path:
    sys.path.insert(0, str(_nb_dir))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import statsmodels.api as sm
from scipy import stats

from geofunctions import utils

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

rng = np.random.default_rng(20260816)

import third_places_neighborhood_retention as tpr
from third_places_neighborhood_retention import *
from third_places_neighborhood_retention import _event_tick
tpr.bind(globals())

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DEMO_PATH = Path(utils.get_path("processed", "statistics", "demographics_2019.geojson"))
SES_PATH = Path(utils.get_path("processed", "statistics", "soceconomindex_2019.geojson"))

DEMO_TABLE = "phd-habidatum.phd_census.sa_demographics_2019"
SES_TABLE = "phd-habidatum.phd_census.sa_socecon_index_2019"
MATCH_TABLE = "phd-habidatum.phd_adm.match_census_2022_2011"
SA_2022_TABLE = "mapinamo-2.isr_geo.statistical_areas_2022"
CENSUS_TABLE = "phd-habidatum.phd_census.all_sa_main_features"
NEIGH_TABLE = "phd-habidatum.phd_datasets.sa_neigborhoods_types_new"
PU_TABLE = "phd-habidatum.phd_models_results.google_3rd_places_detected_pu_20260601"
DWELLS_TABLE = "phd-habidatum.phd_dwells.dwells_pois_home_work_v2"
PROFILES_TABLE = "phd-habidatum.phd_datasets.user_profiles_monthly"
WORK_TABLE = "phd-habidatum.phd_users_locations.work_locations"
RW_TABLE = "phd-habidatum.phd_users_locations.users_remote_work_monthly"
GH_TABLE = "phd-habidatum.phd_adm.geohashes_with_sa_gushdan"

DETECT_BUFFER_M = 500
OUTBOUND_M = 500
# Binary treatment for PSM. Edit treat_rule, re-run this cell, then run_psm_did().
# The rule is not a PS covariate. GPS uses tp_bin, not this flag.
TREAT_FLAG = "has_third_place"
TREAT_LABEL = "Treated (TP activated)"


def treat_rule(sa):
    """Any 0/1 condition. Used by run_psm_did (and Part 2)."""
    return (sa["n_detected"] >= 1).astype(int)
    # return (sa["n_candidates"] >= 1).astype(int)
    # return (sa["n_candidates_atm"] >= 1).astype(int)
    # return (sa["n_detected"] >= 2).astype(int)


# Civic types (ALWAYS_TP_TYPES) are a separate non-commercial candidate count,
# not folded into n_candidates_atm.
ALWAYS_TP_TYPES = ("community_center", "library", "cultural_center")

# Distance bands for "how far when out" (Part 5b). OUTBOUND_M separates local from
# non-local *and* drops named POIs within OUTBOUND_M of work from the outbound
# bucket (office-adjacent cafés etc. are not "left the neighbourhood").
# The FAR_M cut points ask whether the remaining trip left the wider neighbourhood
# (2 km, walk/short ride) or the area entirely (5 km, a commute-scale trip).
FAR_M = (1000, 2000, 5000)
MIN_WORKERS = 10
MIN_WORKER_DAYS = 40
PRE_MONTH = "2020-01-01"
POST_MONTHS = ("2023-01-01", "2023-02-01","2023-03-01")
# Both months enter the long DiD / event study as post. Matching and the wide
# `_post` snapshot use February (the last month), so the matched sample is SAs
# observed in Jan 2020 and Feb 2023.
# Extra pre-period: January 2019 is too thin (~24 users nationally) for any SA to
# clear MIN_WORKERS, so the placebo / lead pools all of 2019 into one SA x window
# row tagged PLACEBO_MONTH. It is never used as the DiD "pre". Coverage is decided
# in Part 3, not assumed here.
PLACEBO_YEAR = 2019
PLACEBO_MONTHS = tuple(f"{PLACEBO_YEAR}-{m:02d}-01" for m in range(1, 13))
PLACEBO_MONTH = f"{PLACEBO_YEAR}-01-01"  # sentinel for the year-pooled 2019 row
TEL_AVIV_SEMEI = 5000

# ---------------------------------------------------------------------------
# Time-of-day windows (local Israel clock)
# ---------------------------------------------------------------------------
# `start_date_time` is already a local wall clock, so no time-zone conversion is applied.
# Checked on January weekdays: work-dwell starts peak at stored hour 8 and the overnight
# activity trough ends at 05:00, both correct on the Israel clock; reading the column as UTC
# would put the work-arrival peak at 10:00 and keep the night trough running to 06:00.
# Caveat for reuse: the offset is fixed at UTC+2 and ignores DST - the stored phase of the
# daily rhythm jumps +1 h within days of the spring switch and back at the autumn switch.
# January pre and January–February post months are outside DST, so stored hour =
# local clock. The 2019 placebo pools the whole year, so summer months sit 1 h off
# in the sub-day windows; all24 is unaffected. Treated and control share the same
# clock, so the placebo is still a valid pre-trend check (slightly noisier at midday).
# Dwell durations are capped at 4 h upstream, and ~4% of dwells cross local midnight,
# so duration is allocated to windows by overlap, not by start hour.
WINDOWS = {              # name: (start_hour, end_hour) local
    "all24": (0, 24),    # reproduces the previous all-day definition
    "daytime": (6, 22),  # drops the night; comparable base for the sub-windows
    "morning": (6, 9),   # commute-departure diagnostic
    "midday": (9, 16),   # office-substitution window
    "break": (11, 15),   # sharper lunch/coffee-break window
    "evening": (16, 22),  # after-work leisure window
}
PRIMARY_WINDOW = "midday"
DID_WINDOWS = ("all24", "morning", "midday", "evening")
PSM_MATCH_WINDOW = "all24"     # wide pre/post snapshot used for 1:1 matching
LOVE_PRE_WINDOW = "midday"     # extra pre/post shares (and d_) on the matching panel
# Minimum observed time per worker-day inside a window, as a fraction of the window
# length. Ping density is low (~3.4 observed hours per worker-day out of 24), so a flat
# 1-hour rule is unreachable in a 3-hour window and wipes out `morning` entirely;
# the SA-level MIN_WORKERS / MIN_WORKER_DAYS thresholds do the real filtering.
MIN_WINDOW_COVERAGE = 0.10
# Named POI must have ≥ this many worker dwells in *each* DiD month (Jan 2020 and
# every post month). Same POI set in pre and post; 2019 lead uses that list.
# Unnamed dwells are unchanged. Treatment (n_detected) is not filtered here.
MIN_POI_DWELLS_PER_MONTH = 5

# Intermediate / output tables
COV_TABLE = "phd-habidatum.phd_datasets.sa_retention_covariates_3300"
OUTCOME_TABLE = "phd-habidatum.phd_datasets.sa_retention_outcomes_win_3300"

WFH_PRE_COL = "wfh_avg_pre"    # Jan 2020 SA mean (diagnostic; not in PS)
WFH_POST_COL = "wfh_avg_post"  # Jan–Mar 2023 SA mean of flag_remote_work_smoothed_max_thresh
WFH_HIGH_COL = "high_wfh_2023"
WFH_HIGH_CUT = 0.2             # same cut as remote_worker in user profiles
WFH_MIN_WORKDAYS = 7

## Helpers (CI / aggregation / matching / DiD)

Non-SQL helpers live in `third_places_neighborhood_retention.py` (imported and bound in the imports cell). SQL builders stay in Parts 2 and 3.


In [ ]:
print("Helpers ready:", tpr.__file__)


## Part 0 — Load 2019 demographics and SES cluster to BigQuery

**Goal.** Put 2019 SES and age layers in BigQuery so Part 1 can join them to 2022 SAs.

**Steps.**
1. Read geojsons (2011 codes `YISHUV_STAT11`, EPSG:2039).
2. Upload attributes only (BQ GEOGRAPHY rejects some nested rings). Skip if `UPLOAD_2019=False`.


In [ ]:
# Re-runnable upload (replace). Skip if tables already exist and you do not need refresh.
UPLOAD_2019 = False  # set True to re-upload 2019 layers

if UPLOAD_2019:
    demo = gpd.read_file(DEMO_PATH)
    ses = gpd.read_file(SES_PATH)
    print(f"demographics_2019: {len(demo):,} rows, crs={demo.crs}")
    print(f"soceconomindex_2019: {len(ses):,} rows, crs={ses.crs}")
    utils.upload_table_to_bq(
        demo.drop(columns=["geometry"]),
        "phd_census",
        "sa_demographics_2019",
        geometry_column=None,
        if_table_exist="replace",
    )
    utils.upload_table_to_bq(
        ses.drop(columns=["geometry"]),
        "phd_census",
        "sa_socecon_index_2019",
        geometry_column=None,
        if_table_exist="replace",
    )
else:
    print("Skipped upload (UPLOAD_2019=False).")


## Part 1 — Crosswalk 2011 → 2022 and SA covariates

**Goal.** One 2022-SA covariate table for matching: SES, working-age share, CBD distance, amenity weight.

**Steps.**
1. Map 2011 → 2022 via `phd_adm.match_census_2022_2011` (column typo `yeshuv_stat_2022`) plus identity rows for unchanged codes.
2. Collapse: pop-weighted age share; pop-weighted mean eshkol (nearest int).
3. Join CBD / amenity covariates; write `COV_TABLE`; check missingness.


In [ ]:
run_cov = False
sql_cov = f"""
CREATE OR REPLACE TABLE `{COV_TABLE}` AS
WITH matching AS (
  SELECT CAST(b.yishuv_stat11 AS STRING) AS stat_area_11,
         CAST(b.yeshuv_stat_2022 AS STRING) AS stat_area_2022
  FROM `{MATCH_TABLE}` b
  UNION DISTINCT
  SELECT CAST(yishuv_stat_2022 AS STRING), CAST(yishuv_stat_2022 AS STRING)
  FROM `{SA_2022_TABLE}`
),
demo AS (
  SELECT
    CAST(yishuv_stat11 AS STRING) AS yishuv_stat11,
    pop_total,
    main_function_txt,
    (
      IFNULL(age_20_24, 0) + IFNULL(age_25_29, 0) + IFNULL(age_30_34, 0)
      + IFNULL(age_35_39, 0) + IFNULL(age_40_44, 0) + IFNULL(age_45_49, 0)
      + IFNULL(age_50_54, 0) + IFNULL(age_55_59, 0)
    ) AS pop_20_59,
    SAFE_DIVIDE(
      IFNULL(age_20_24, 0) + IFNULL(age_25_29, 0) + IFNULL(age_30_34, 0)
      + IFNULL(age_35_39, 0) + IFNULL(age_40_44, 0) + IFNULL(age_45_49, 0)
      + IFNULL(age_50_54, 0) + IFNULL(age_55_59, 0),
      NULLIF(pop_total, 0)
    ) AS age20_59_pcnt
  FROM `{DEMO_TABLE}`
),
ses AS (
  SELECT CAST(yishuv_stat11 AS STRING) AS yishuv_stat11,
         eshkol_madad2019
  FROM `{SES_TABLE}`
),
joined_11 AS (
  SELECT
    m.stat_area_2022 AS yishuv_stat_2022,
    m.stat_area_11 AS yishuv_stat11,
    d.pop_total,
    d.pop_20_59,
    d.age20_59_pcnt,
    d.main_function_txt,
    s.eshkol_madad2019
  FROM matching m
  LEFT JOIN demo d ON d.yishuv_stat11 = m.stat_area_11
  LEFT JOIN ses s ON s.yishuv_stat11 = m.stat_area_11
),
agg_2022 AS (
  SELECT
    yishuv_stat_2022,
    SUM(IFNULL(pop_total, 0)) AS pop_total_2019,
    SAFE_DIVIDE(SUM(IFNULL(pop_20_59, 0)), NULLIF(SUM(IFNULL(pop_total, 0)), 0)) AS age20_59_pcnt,
    -- pop-weighted mean of cluster, then round to nearest int
    CAST(ROUND(SAFE_DIVIDE(
      SUM(IFNULL(eshkol_madad2019, 0) * IFNULL(pop_total, 0)),
      NULLIF(SUM(CASE WHEN eshkol_madad2019 IS NOT NULL THEN IFNULL(pop_total, 0) END), 0)
    )) AS INT64) AS eshkol_madad2019,
    COUNT(*) AS n_2011_parts,
    COUNTIF(eshkol_madad2019 IS NOT NULL) AS n_with_eshkol,
    ANY_VALUE(main_function_txt) AS main_function_txt_2019
  FROM joined_11
  GROUP BY yishuv_stat_2022
)
SELECT
  a.yishuv_stat_2022,
  a.pop_total_2019,
  a.age20_59_pcnt,
  a.eshkol_madad2019,
  a.n_2011_parts,
  a.n_with_eshkol,
  a.main_function_txt_2019,
  CAST(c.semel_yishuv AS INT64) AS semel_yishuv,
  c.main_function_txt AS main_function_txt_2022,
  nt.distance_to_azrieli / 1000.0 AS dist_to_cbd_km,
  nt.amenities_1km_area_wght,
  nt.new_neighborhood_type,
  (CAST(c.semel_yishuv AS INT64) = {TEL_AVIV_SEMEI}) AS flag_tel_aviv
FROM agg_2022 a
INNER JOIN `{NEIGH_TABLE}` nt
  ON CAST(nt.yishuv_stat_2022 AS STRING) = a.yishuv_stat_2022
LEFT JOIN `{CENSUS_TABLE}` c
  ON CAST(c.yishuv_stat_2022 AS STRING) = a.yishuv_stat_2022
where 	pop_total_2019>0
"""
if run_cov:
  utils.read_query_to_dataframe(sql_cov)
# BigQuery returns rows in arbitrary order; sorting pins the summation order of the
# group means below. Rebuilding the aggregate tables still moves the last few digits
# (parallel float SUM in BigQuery is not associative), so runs agree to ~1e-6, not exactly.
df_cov = (utils.read_query_to_dataframe(f"SELECT * FROM `{COV_TABLE}`")
          .sort_values("yishuv_stat_2022", ignore_index=True))

## Part 2 — Treatment: PU third-place activation

**Goal.** Assign each SA a binary treatment flag and a continuous intensity for DiD.

**Steps.**
1. Count PU candidates / detections inside the `DETECT_BUFFER_M` centroid buffer only (no in-SA `GREATEST`).
2. `n_detected` = PU `prediction_third_place = 1`. `n_candidates_atm` = atmosphere match only (`flag_atmosphere_suits_rw = 1`).
3. Set `TREAT_FLAG` from `treat_rule` (config; default `n_detected >= 1` → `has_third_place`).
4. Attach `tp_intensity`, `cand_intensity`, `atm_intensity` (count × 1000 / 2019 pop).
5. Bin `tp_intensity` into `tp_bin` for multinomial IPW: 0 if intensity is 0; 1 if positive and ≤ the 75th percentile of **non-zero** values; 2 if above that percentile (right tail).

`n_detected`, `n_candidates`, and `n_candidates_atm` are buffer counts only
(`ST_DWITHIN` of the SA centroid, radius `DETECT_BUFFER_M`). In-SA polygon counts are not used.
Main dosage is `tp_intensity` = `n_detected * 1000 / pop_total_2019` (detected third places per 1,000 residents in 2019). Continuous CBGPS on this dosage fails (zero-inflation and a long right tail), so the GPS/TWFE path uses `tp_bin` and multinomial IPW instead. Do not divide the ratio by 1,000 a second time: that scaled intensity to ~10⁻⁶ and exploded the DiD coefficients.

PSM grouping uses `TREAT_FLAG` from `treat_rule` (config). Default is `n_detected >= 1`
→ `has_third_place`. Change the rule (or pass a callable to `run_psm_did`) to group on
amenity stock or another 0/1 condition. GPS/TWFE still uses `tp_bin`, not this flag.

Dosages, same per-1,000 scaling:
- `cand_intensity` — all PU candidates (`n_candidates`)
- `atm_intensity` — atmosphere match only (`n_candidates_atm`: `flag_atmosphere_suits_rw = 1`)
- `comm_cand_intensity` — commercial candidates only (`n_candidates_comm`: `primarytype` not civic)
- `noncomm_cand_intensity` — non-commercial (civic) candidates (`n_candidates_noncomm`: museum / library / community_center / cultural_center)

`always_tp_sql` lives in this cell so Part 2 runs without re-running config.


In [ ]:
def always_tp_sql(col):
    """SQL predicate: civic types counted with atmosphere, not with PU detections."""
    types_sql = ", ".join(f"'{t}'" for t in ALWAYS_TP_TYPES)
    return (
        "REGEXP_REPLACE(LOWER(REPLACE(REPLACE(COALESCE("
        f"{col}, ''), '-', '_'), ' ', '_')), r'[^a-z0-9_]', '') IN ({types_sql})"
    )


CIVIC_CAND_TYPES = ("museum", "library", "community_center", "cultural_center")


def type_in_sql(col, types):
    types_sql = ", ".join(f"'{t}'" for t in types)
    return (
        "REGEXP_REPLACE(LOWER(REPLACE(REPLACE(COALESCE("
        f"{col}, ''), '-', '_'), ' ', '_')), r'[^a-z0-9_]', '') IN ({types_sql})"
    )


def build_treat_sql(detect_buffer_m):
    is_atm_only = "COALESCE(CAST(p.flag_atmosphere_suits_rw AS INT64), 0) = 1"
    is_civic = type_in_sql("p.primarytype", CIVIC_CAND_TYPES)
    is_comm = f"NOT ({is_civic})"
    return f"""
WITH sa AS (
  SELECT CAST(yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
         geometry AS sa_geom
  FROM `{CENSUS_TABLE}`
),
buf AS (
  SELECT sa.yishuv_stat_2022,
         COUNT(*) AS n_candidates,
         COUNTIF({is_atm_only}) AS n_candidates_atm,
         COUNTIF({is_comm}) AS n_candidates_comm,
         COUNTIF({is_civic}) AS n_candidates_noncomm,
         COUNTIF(p.prediction_third_place = 1) AS n_detected
  FROM sa
  JOIN `{PU_TABLE}` p
    ON p.geometry IS NOT NULL
   AND ST_DWITHIN(ST_CENTROID(sa.sa_geom), p.geometry, {detect_buffer_m})
  GROUP BY 1
)
SELECT
  cov.*,
  COALESCE(buf.n_candidates, 0) AS n_candidates,
  COALESCE(buf.n_candidates_atm, 0) AS n_candidates_atm,
  COALESCE(buf.n_candidates_comm, 0) AS n_candidates_comm,
  COALESCE(buf.n_candidates_noncomm, 0) AS n_candidates_noncomm,
  COALESCE(buf.n_detected, 0) AS n_detected,
  SAFE_DIVIDE(
    COALESCE(buf.n_detected, 0),
    NULLIF(COALESCE(buf.n_candidates, 0), 0)
  ) AS share_detected
FROM `{COV_TABLE}` cov
LEFT JOIN buf ON buf.yishuv_stat_2022 = cov.yishuv_stat_2022
"""


def build_sa_wfh_sql(months, out_col, n_col):
    """SA-mean WFH over the given months."""
    months_sql = ", ".join(f"DATE '{m}'" for m in months)
    return f"""
SELECT
  CAST(g.yishuv_stat_2022 AS STRING) AS yishuv_stat_2022,
  AVG(rw.flag_remote_work_smoothed_max_thresh) AS {out_col},
  COUNT(*) AS {n_col}
FROM `{RW_TABLE}` rw
INNER JOIN `{WORK_TABLE}` w
  ON CAST(rw.identifier AS STRING) = CAST(w.identifier AS STRING)
 AND DATE(rw.month) = DATE(w.month)
INNER JOIN `{GH_TABLE}` g
  ON g.geohash = w.geohash_home
WHERE DATE(rw.month) IN ({months_sql})
  AND w.geohash_home IS NOT NULL
  AND rw.flag_remote_work_smoothed_max_thresh IS NOT NULL
  AND rw.total_work_days >= {WFH_MIN_WORKDAYS}
GROUP BY 1
"""


def load_sa_wfh(months, out_col, n_col):
    df = utils.read_query_to_dataframe(build_sa_wfh_sql(months, out_col, n_col))
    df["yishuv_stat_2022"] = df["yishuv_stat_2022"].astype(str)
    df[out_col] = pd.to_numeric(df[out_col], errors="coerce")
    df[n_col] = pd.to_numeric(df[n_col], errors="coerce").fillna(0).astype(int)
    return df


def load_treatment(detect_buffer_m, flag_col=None, rule=None):
    sa = (utils.read_query_to_dataframe(build_treat_sql(detect_buffer_m))
          .sort_values("yishuv_stat_2022", ignore_index=True))
    sa["yishuv_stat_2022"] = sa["yishuv_stat_2022"].astype(str)
    sa = attach_intensities(sa)
    sa = apply_treat_flag(sa, flag_col=flag_col, rule=rule)
    pre = load_sa_wfh((PRE_MONTH,), WFH_PRE_COL, "n_wfh_users")
    post = load_sa_wfh(POST_MONTHS, WFH_POST_COL, "n_wfh_users_post")
    sa = sa.merge(pre, on="yishuv_stat_2022", how="left")
    sa = sa.merge(post, on="yishuv_stat_2022", how="left")
    sa[WFH_HIGH_COL] = np.where(
        sa[WFH_POST_COL].notna(),
        (sa[WFH_POST_COL] >= WFH_HIGH_CUT).astype(int),
        np.nan,
    )
    print(f"Jan 2020 SA-mean WFH (`{WFH_PRE_COL}`): "
          f"{sa[WFH_PRE_COL].notna().sum():,} SAs, "
          f"{sa[WFH_PRE_COL].isna().sum():,} missing")
    print(sa[[WFH_PRE_COL, "n_wfh_users"]].describe().round(3))
    print(f"2023 SA-mean WFH (`{WFH_POST_COL}`, months={POST_MONTHS}): "
          f"{sa[WFH_POST_COL].notna().sum():,} SAs, "
          f"{sa[WFH_POST_COL].isna().sum():,} missing")
    print(sa[[WFH_POST_COL, "n_wfh_users_post"]].describe().round(3))
    print(f"{WFH_HIGH_COL} = 1{{{WFH_POST_COL} >= {WFH_HIGH_CUT}}}")
    print(sa[WFH_HIGH_COL].value_counts(dropna=False).rename("n_SA"))
    return sa


df_sa = load_treatment(DETECT_BUFFER_M)
print(f"Treatment flag={TREAT_FLAG} from treat_rule")
print(df_sa[TREAT_FLAG].value_counts().rename("n_SA"))
print(f"SAs: {len(df_sa):,}  P({TREAT_FLAG})={df_sa[TREAT_FLAG].mean():.3f}")
print(f"n_detected mean={df_sa.n_detected.mean():.3f} max={df_sa.n_detected.max()}")
print(f"n_candidates mean={df_sa.n_candidates.mean():.3f} max={df_sa.n_candidates.max()}")
print(f"n_candidates_atm (atmosphere flag only) mean="
      f"{df_sa.n_candidates_atm.mean():.3f} max={df_sa.n_candidates_atm.max()}")
print(f"n_candidates_comm (not civic) mean="
      f"{df_sa.n_candidates_comm.mean():.3f} max={df_sa.n_candidates_comm.max()}")
print(f"n_candidates_noncomm (civic) mean="
      f"{df_sa.n_candidates_noncomm.mean():.3f} max={df_sa.n_candidates_noncomm.max()}")
for col, lab in [
    ("tp_intensity", "detected"),
    ("cand_intensity", "candidates"),
    ("atm_intensity", "atmosphere flag only"),
    ("comm_cand_intensity", "commercial candidates"),
    ("noncomm_cand_intensity", "non-commercial (civic) candidates"),
]:
    print(f"{col} ({lab}) mean={df_sa[col].mean():.3f} "
          f"p50={df_sa[col].median():.3f} max={df_sa[col].max():.3f}")

In [ ]:
# Categorical dose for multinomial IPW (no BQ). Uses existing df_sa.tp_intensity.
# 0 = no detected TP; 1 = non-zero up to p75 of positives; 2 = right tail.


df_sa, TP_BIN_P75 = assign_tp_bin(df_sa)
print(f"Non-zero tp_intensity 75th percentile = {TP_BIN_P75:.4f} per 1,000 residents")
print("tp_bin counts:")
print(df_sa["tp_bin"].value_counts().sort_index().rename("n_SA"))
print(df_sa.groupby("tp_bin")["tp_intensity"].agg(["count", "min", "median", "max"]))


# Changes can be done after this part

## Part 3 — Weekday local-share outcomes by time-of-day window

**Goal.** SA × month × window weekday **local shares** (visits and duration), for workers with a known work location.

**Steps.**
1. Restrict to profiles ∩ known work locations; weekdays; home SA = `home_yishuv_stat_2022`.
2. Allocate dwell duration by overlap to `all24`, `morning`, `midday`, and `evening`.
3. Build 12 local-share columns (6 families × visits / hours); keep SAs above `MIN_WORKERS` / `MIN_WORKER_DAYS`.
4. Build long panel (all months) and wide pre/post (Jan 2020 vs last post month) for matching; attach midday pre-shares for the love plot.

Sample: `user_profiles_monthly` ∩ `work_locations` (known work) in the analysis months.
Home SA = `home_yishuv_stat_2022`. Weekdays (Sun–Thu) only, on the **local** date.

**Radii.** Home local = `dist_to_home < OUTBOUND_M` (500 m). Near work = `dist_to_work <= WORK_NEAR_M` (**250 m**). `flag_work` stays geohash equality.

**Named POIs.** A `poi_id` enters named-POI / commercial / PU-candidate shares only if it has at least `MIN_POI_DWELLS_PER_MONTH` (5) dwells by sample workers in **every** DiD month (Jan 2020 and each post month). One-off venues are treated as unnamed. 2019 uses the same eligible list. Treatment counts in Part 2 are unchanged.

**Twelve shares** (each is local / denominator unless noted; `SAFE_DIVIDE` if empty):
1. Among **all named POIs** (`poi_id` not null) — visits and hours. Primary retention outcome.
2. Same after dropping `dist_to_work <= 250` (office-adjacent POIs leave both sides).
3. Among **commercial named POIs** (named POI whose `pois_category` is not civic: museum, library, community_center, cultural_center).
4. Among **PU candidates** (dwell `id` in the PU table) — share that are local (mechanism).
5. Among **all dwells** (named + unnamed / no `poi_id`) — WFH diagnostic (includes home).
6. Among **not-home dwells** (`NOT flag_home`, including unnamed and work) — local vs outbound when not at home.

Dropped (not identified / not the claim): `share_local_place` (duplicates 1); `share_local_cand_nowork` (composition, not local vs outbound); `share_local_tp` (controls structurally ~0).

Windows: **all24**, **morning** (6–9), **midday** (9–16), **evening** (16–22) → 48 DiD rows (12 metrics × 4 windows). `run_psm_did` fits treat × post on every share in `SHARE_OUTCOME_COLS`.

Pre = Jan 2020. Post = Jan–Mar 2023 (matching uses the last post month). Placebo / lead = calendar year 2019, pooled across months into one SA × window row.


In [ ]:
# Time-of-day windows for the local-share rebuild (12 metrics × 4 windows).
WINDOWS = {
    "all24": (0, 24),
    "morning": (6, 9),    # commute-departure
    "midday": (9, 16),    # office-substitution
    "evening": (16, 22),  # after-work leisure
}
WORK_NEAR_M = 250
CIVIC_TYPES = ("museum", "library", "community_center", "cultural_center")

ALL_MONTHS = [*PLACEBO_MONTHS, PRE_MONTH, *POST_MONTHS]
PANEL_MONTHS = [PLACEBO_MONTH, PRE_MONTH, *POST_MONTHS]


def civic_cat_sql(col):
    types_sql = ", ".join(f"'{t}'" for t in CIVIC_TYPES)
    return (
        "REGEXP_REPLACE(LOWER(REPLACE(REPLACE(COALESCE("
        f"{col}, ''), '-', '_'), ' ', '_')), r'[^a-z0-9_]', '') IN ({types_sql})"
    )


def build_outcomes_sql(outbound_m, outcome_table, work_near_m=None):
    work_near_m = WORK_NEAR_M if work_near_m is None else work_near_m
    months_list = "', '".join(ALL_MONTHS)
    did_months = [PRE_MONTH, *POST_MONTHS]
    did_months_list = "', '".join(did_months)
    n_did_months = len(did_months)
    win_rows = "\n  UNION ALL ".join(
        f"SELECT '{name}' AS win, {h0} AS h0, {h1} AS h1, "
        f"{0.0 if name == 'all24' else round(MIN_WINDOW_COVERAGE * (h1 - h0), 4)} AS min_h"
        for name, (h0, h1) in WINDOWS.items()
    )
    is_civic = civic_cat_sql("d.pois_category")
    return f"""
CREATE OR REPLACE TABLE `{outcome_table}` AS
WITH workers AS (
  SELECT DISTINCT
    CAST(p.identifier AS STRING) AS identifier,
    DATE(p.month) AS month,
    CAST(p.home_yishuv_stat_2022 AS STRING) AS yishuv_stat_2022
  FROM `{PROFILES_TABLE}` p
  INNER JOIN `{WORK_TABLE}` w
    ON CAST(w.identifier AS STRING) = CAST(p.identifier AS STRING)
   AND DATE(w.month) = DATE(p.month)
  WHERE DATE(p.month) IN ('{months_list}')
    AND p.home_yishuv_stat_2022 IS NOT NULL
),
dwells AS (
  SELECT
    CAST(d.identifier AS STRING) AS identifier,
    DATE(d.month) AS month,
    DATETIME(d.start_date_time) AS local_start,
    DATETIME_ADD(
      DATETIME(d.start_date_time),
      INTERVAL CAST(ROUND(d.duration_hours * 3600) AS INT64) SECOND
    ) AS local_end,
    COALESCE(d.flag_home, FALSE) AS flag_home,
    COALESCE(d.flag_work, FALSE) AS flag_work,
    d.dist_to_home,
    d.dist_to_work,
    CAST(d.id AS STRING) AS poi_id,
    d.pois_category,
    w.yishuv_stat_2022,
    (pu.id IS NOT NULL) AS is_candidate,
    COALESCE(pu.is_detected, FALSE) AS is_detected
  FROM `{DWELLS_TABLE}` d
  INNER JOIN workers w
    ON w.identifier = CAST(d.identifier AS STRING)
   AND w.month = DATE(d.month)
  LEFT JOIN (
    SELECT
      CAST(id AS STRING) AS id,
      LOGICAL_OR(prediction_third_place = 1) AS is_detected
    FROM `{PU_TABLE}`
    WHERE id IS NOT NULL
    GROUP BY 1
  ) pu ON pu.id = CAST(d.id AS STRING)
  WHERE DATE(d.month) IN ('{months_list}')
    AND d.duration_hours IS NOT NULL
    AND d.duration_hours > 0
    AND d.start_date_time IS NOT NULL
),
poi_ok AS (
  -- Named POIs with ≥ MIN_POI_DWELLS_PER_MONTH worker dwells in every DiD month.
  SELECT poi_id
  FROM (
    SELECT poi_id, month, COUNT(*) AS n_dwells
    FROM dwells
    WHERE poi_id IS NOT NULL AND poi_id != ''
      AND month IN ('{did_months_list}')
    GROUP BY poi_id, month
  )
  GROUP BY poi_id
  HAVING COUNTIF(n_dwells >= {MIN_POI_DWELLS_PER_MONTH}) = {n_did_months}
),
windows AS (
  {win_rows}
),
alloc AS (
  -- Overlap allocation: hours of the dwell that fall inside each window,
  -- evaluated on the local date and the following one (cross-midnight dwells).
  SELECT
    d.yishuv_stat_2022,
    d.month,
    d.identifier,
    w.win,
    dd AS local_date,
    GREATEST(0, DATETIME_DIFF(
      LEAST(d.local_end, DATETIME_ADD(DATETIME(dd), INTERVAL w.h1 HOUR)),
      GREATEST(d.local_start, DATETIME_ADD(DATETIME(dd), INTERVAL w.h0 HOUR)),
      SECOND)) / 3600.0 AS dur_h,
    (d.poi_id IS NOT NULL AND d.poi_id != '' AND ok.poi_id IS NOT NULL) AS is_poi,
    (d.dist_to_home IS NOT NULL AND d.dist_to_home < {outbound_m}) AS is_local,
    (d.dist_to_work IS NOT NULL AND d.dist_to_work <= {work_near_m}) AS is_near_work,
    (NOT d.flag_home AND NOT d.flag_work) AS is_not_hw,
    (NOT COALESCE(d.flag_home, FALSE)) AS is_nothome,
    (
      d.poi_id IS NOT NULL AND d.poi_id != '' AND ok.poi_id IS NOT NULL
      AND NOT ({is_civic})
    ) AS is_commercial,
    (COALESCE(d.is_candidate, FALSE) AND ok.poi_id IS NOT NULL) AS is_candidate,
    (COALESCE(d.is_detected, FALSE) AND ok.poi_id IS NOT NULL) AS is_detected
  FROM dwells d
  LEFT JOIN poi_ok ok ON ok.poi_id = d.poi_id
  CROSS JOIN windows w
  CROSS JOIN UNNEST([DATE(d.local_start), DATE_ADD(DATE(d.local_start), INTERVAL 1 DAY)]) AS dd
),
alloc_wd AS (
  SELECT * FROM alloc
  WHERE dur_h > 0
    AND EXTRACT(DAYOFWEEK FROM local_date) IN (1, 2, 3, 4, 5)  -- Sun..Thu
),
user_day AS (
  SELECT
    yishuv_stat_2022,
    month,
    win,
    identifier,
    local_date,
    SUM(dur_h) AS dur_total
  FROM alloc_wd
  GROUP BY 1, 2, 3, 4, 5
),
user_day_ok AS (
  SELECT ud.*
  FROM user_day ud
  JOIN windows w USING (win)
  WHERE ud.dur_total >= w.min_h
),
alloc_ok AS (
  SELECT a.*
  FROM alloc_wd a
  JOIN user_day_ok u
    USING (yishuv_stat_2022, month, win, identifier, local_date)
),
alloc_period AS (
  -- January 2019 never clears MIN_WORKERS on its own. Remap every 2019 user-day
  -- onto PLACEBO_MONTH so sa_month_win pools the calendar year before the SA filter.
  SELECT
    * EXCEPT (month),
    IF(EXTRACT(YEAR FROM month) = {PLACEBO_YEAR},
       DATE '{PLACEBO_MONTH}',
       month) AS month
  FROM alloc_ok
),
sa_month_win AS (
  SELECT
    yishuv_stat_2022,
    month,
    win,
    COUNT(DISTINCT identifier) AS n_workers,
    COUNT(DISTINCT CONCAT(identifier, '|', CAST(local_date AS STRING))) AS n_worker_days,
    SUM(dur_h) AS dur_total,
    SAFE_DIVIDE(
      SUM(dur_h),
      COUNT(DISTINCT CONCAT(identifier, '|', CAST(local_date AS STRING)))
    ) AS obs_h_per_day,
    -- 1. among all named POIs, share with dist_to_home < OUTBOUND_M
    SAFE_DIVIDE(COUNTIF(is_poi AND is_local), COUNTIF(is_poi)) AS share_local_poi_n,
    SAFE_DIVIDE(
      SUM(IF(is_poi AND is_local, dur_h, 0)),
      SUM(IF(is_poi, dur_h, 0))
    ) AS share_local_poi_h,
    -- 2. same after dropping POIs within WORK_NEAR_M of work
    SAFE_DIVIDE(
      COUNTIF(is_poi AND NOT is_near_work AND is_local),
      COUNTIF(is_poi AND NOT is_near_work)
    ) AS share_local_poi_nowork_n,
    SAFE_DIVIDE(
      SUM(IF(is_poi AND NOT is_near_work AND is_local, dur_h, 0)),
      SUM(IF(is_poi AND NOT is_near_work, dur_h, 0))
    ) AS share_local_poi_nowork_h,
    -- 3. among commercial named POIs (not civic)
    SAFE_DIVIDE(
      COUNTIF(is_commercial AND is_local),
      COUNTIF(is_commercial)
    ) AS share_local_comm_n,
    SAFE_DIVIDE(
      SUM(IF(is_commercial AND is_local, dur_h, 0)),
      SUM(IF(is_commercial, dur_h, 0))
    ) AS share_local_comm_h,
    -- 4. among PU candidates, share with dist_to_home < OUTBOUND_M
    SAFE_DIVIDE(COUNTIF(is_candidate AND is_local), COUNTIF(is_candidate)) AS share_local_cand_n,
    SAFE_DIVIDE(
      SUM(IF(is_candidate AND is_local, dur_h, 0)),
      SUM(IF(is_candidate, dur_h, 0))
    ) AS share_local_cand_h,
    -- 5. all local visits / all dwells (named + unnamed)
    SAFE_DIVIDE(COUNTIF(is_local), COUNT(*)) AS share_local_all_n,
    SAFE_DIVIDE(
      SUM(IF(is_local, dur_h, 0)),
      SUM(dur_h)
    ) AS share_local_all_h,
    -- 6. not-home local visits / all not-home dwells (named + unnamed)
    SAFE_DIVIDE(
      COUNTIF(is_nothome AND is_local),
      COUNTIF(is_nothome)
    ) AS share_local_nothome_n,
    SAFE_DIVIDE(
      SUM(IF(is_nothome AND is_local, dur_h, 0)),
      SUM(IF(is_nothome, dur_h, 0))
    ) AS share_local_nothome_h
  FROM alloc_period
  GROUP BY 1, 2, 3
)
SELECT *
FROM sa_month_win
WHERE n_workers >= {MIN_WORKERS}
  AND n_worker_days >= {MIN_WORKER_DAYS}
"""


def load_outcomes(outbound_m, outcome_table, rebuild=True, work_near_m=None):
    if rebuild:
        utils.read_query_to_dataframe(
            build_outcomes_sql(outbound_m, outcome_table, work_near_m=work_near_m)
        )
    win = utils.read_query_to_dataframe(f"SELECT * FROM `{outcome_table}`")
    win["month"] = pd.to_datetime(win["month"])
    return win.sort_values(["yishuv_stat_2022", "win", "month"], ignore_index=True)


df_win = load_outcomes(OUTBOUND_M, OUTCOME_TABLE, rebuild=True)

print("SAs passing thresholds, by window and month:")
print(df_win.pivot_table(index="win", columns="month", values="yishuv_stat_2022", aggfunc="nunique"))
print("\nObserved hours per worker-day (ping-density diagnostic):")
print(df_win.pivot_table(index="win", columns="month", values="obs_h_per_day", aggfunc="mean").round(2))
print("\nMean local POI hour-share (metric 1):")
print(df_win.pivot_table(index="win", columns="month", values="share_local_poi_h", aggfunc="mean").round(3))
print("\nMean local / all dwells, visits (named + unnamed):")
print(df_win.pivot_table(index="win", columns="month", values="share_local_all_n", aggfunc="mean").round(3))
print("\nMean local / not-home dwells, visits (named + unnamed):")
print(df_win.pivot_table(index="win", columns="month", values="share_local_nothome_n", aggfunc="mean").round(3))

MONTHS_IN_PANEL = sorted(str(pd.Timestamp(m).date()) for m in df_win["month"].unique())
MONTHS_MISSING = [m for m in PANEL_MONTHS if m not in MONTHS_IN_PANEL]
PLACEBO_OK = PLACEBO_MONTH in MONTHS_IN_PANEL
print(f"\nMonths in panel: {MONTHS_IN_PANEL}")
if MONTHS_MISSING:
    print(f"Requested but absent (no SA clears MIN_WORKERS={MIN_WORKERS} and "
          f"MIN_WORKER_DAYS={MIN_WORKER_DAYS}): {MONTHS_MISSING}")
print(f"Pre-trend lead (calendar {PLACEBO_YEAR}, pooled) available: {PLACEBO_OK}")
print(f"Home radius OUTBOUND_M={OUTBOUND_M} m; work cut WORK_NEAR_M={WORK_NEAR_M} m")
print(
    f"Named POIs: ≥{MIN_POI_DWELLS_PER_MONTH} worker dwells in each of "
    f"{[PRE_MONTH, *POST_MONTHS]} (same ids in 2019 lead)"
)

df_out = df_win[df_win["win"] == "all24"].drop(columns=["win"]).reset_index(drop=True)

In [ ]:
# Wide: pre (Jan 2020) + matching post (last POST_MONTHS). Other post months
# stay in the long panel. Use Timestamps — after to_datetime, string filters
# like "2023-01-01" do not match.
df_win["month"] = pd.to_datetime(df_win["month"])
PRE_TS = pd.Timestamp(PRE_MONTH)
POST_TS = [pd.Timestamp(x) for x in POST_MONTHS]
MATCH_POST_TS = POST_TS[-1]
POST_2023 = MATCH_POST_TS
PLACEBO_TS = pd.Timestamp(PLACEBO_MONTH)
PRE_LABEL = PRE_TS.strftime("%b %Y")
MATCH_POST_LABEL = MATCH_POST_TS.strftime("%b %Y")
POST_LABEL = (
    MATCH_POST_LABEL if len(POST_TS) == 1
    else f"{POST_TS[0].strftime('%b')}–{POST_TS[-1].strftime('%b %Y')}"
)

# 12 local-share outcomes (windows live on the long table as `win`). All enter DiD.
SHARE_OUTCOME_COLS = [
    "share_local_poi_n",
    "share_local_poi_h",
    "share_local_poi_nowork_n",
    "share_local_poi_nowork_h",
    "share_local_comm_n",
    "share_local_comm_h",
    "share_local_cand_n",
    "share_local_cand_h",
    "share_local_all_n",
    "share_local_all_h",
    "share_local_nothome_n",
    "share_local_nothome_h",
]
SHARE_OUTCOME_LABELS = {
    "share_local_poi_n": "Local | all POIs, visits",
    "share_local_poi_h": "Local | all POIs, hours",
    "share_local_poi_nowork_n": "Local | POIs not near work, visits",
    "share_local_poi_nowork_h": "Local | POIs not near work, hours",
    "share_local_comm_n": "Local | commercial POIs, visits",
    "share_local_comm_h": "Local | commercial POIs, hours",
    "share_local_cand_n": "Local | PU candidates, visits",
    "share_local_cand_h": "Local | PU candidates, hours",
    "share_local_all_n": "Local | all dwells, visits",
    "share_local_all_h": "Local | all dwells, hours",
    "share_local_nothome_n": "Local | not-home dwells, visits",
    "share_local_nothome_h": "Local | not-home dwells, hours",
}
OUTCOME_COLS = list(SHARE_OUTCOME_COLS) + ["obs_h_per_day"]
share_cols = OUTCOME_COLS

# One PSM for all 12 DiDs. y = TREAT_FLAG; X never includes intensities or post shares.
PS_COVARIATES = [
    "eshkol_madad2019",
    "age20_59_pcnt",
    "dist_to_cbd_km",
    "amenities_1km_area_wght",
    "share_local_poi_h_pre",
]
CALIPER_SD = 0.1  # on logit PS

MAIN_OUTCOMES = [(c, SHARE_OUTCOME_LABELS[c]) for c in SHARE_OUTCOME_COLS]
OUT_LABELS = dict(MAIN_OUTCOMES)
SA_KEYS = [
    "yishuv_stat_2022", TREAT_FLAG, "n_detected", "n_candidates",
    "n_candidates_atm", "n_candidates_atm_only", "n_candidates_comm",
    "n_candidates_noncomm", "share_detected",
    "tp_intensity", "cand_intensity", "atm_intensity", "atm_only_intensity",
    "comm_cand_intensity", "noncomm_cand_intensity", "share_tp", "nontp_candidate_intensity",
    "tp_bin", "ipw_weight",
    "semel_yishuv", "flag_tel_aviv", "pair_id", "ps", "logit_ps",
    "pop_total_2019",
    *PS_COVARIATES,
]


panel, panel_m, df_out = build_matching_panel(df_sa, df_win)
print(f"pre SAs: {df_out.loc[df_out['month']==PRE_TS, 'yishuv_stat_2022'].nunique():,}  "
      f"post SAs ({MATCH_POST_LABEL}): "
      f"{df_out.loc[df_out['month']==MATCH_POST_TS, 'yishuv_stat_2022'].nunique():,}")
print(f"df_sa: {len(df_sa):,}  month dtype: {df_out['month'].dtype}")
print("month value counts:\n", df_out["month"].value_counts().sort_index())
print(f"Panel SAs with pre + post: {len(panel):,}")
print(f"Matching sample (non-null covariates): {len(panel_m):,}")
print("PS covariates:", PS_COVARIATES)
print(panel_m[TREAT_FLAG].value_counts())
print(panel_m[[c for c in PS_COVARIATES if c in panel_m.columns]].describe().round(3))
_desc_cols = [c for c in (
    WFH_PRE_COL, "share_local_poi_h_pre", "share_local_poi_h_post", "d_share_local_poi_h",
    "share_local_poi_h_midday_pre",
    "share_local_all_n_pre", "share_local_all_h_pre",
    "share_local_nothome_n_pre", "share_local_nothome_h_pre",
) if c in panel_m.columns]

## Part 4 — Propensity-score matching (PSM)

**Goal.** Pair treated and control SAs that look the same on confounders, keeping the third-place contrast. One match for all local-share DiDs.

**Steps.**
1. Choose the binary treatment with `treat_rule` (config), then `run_psm_did()`. That rebuilds the flag, rematches 1:1, and prints Table 1. No Part 2 re-run. Optional: `run_psm_did(lambda sa: sa["n_candidates_atm"] >= 1)`.
2. Drop SAs missing PS covariates; z-score X; fit logit P(`TREAT_FLAG` = 1 | X).
3. 1:1 nearest neighbour on logit PS (caliper = 0.1 × SD), without replacement.
4. Love plot: Panel A = PS covariates + intensities; Panel B = all Jan 2020 outcome shares before vs after matching.

Third-place activation is the **treatment**, so it is the difference we keep between
test and control. Matching is only on **confounders**. `n_detected`, `n_candidates`,
`tp_intensity`, `cand_intensity` and `atm_intensity` never enter the propensity score;
putting them in would match away the contrast the DiD is built on.

The logit estimates P(`TREAT_FLAG` = 1 | X) in order to pair treated SAs with
control SAs that look the same on X. The 16 mobility shares are **outcomes**, not a
second `y`. Only the all-day metric-1 duration pre-share is in X.

X (z-scored):
- `eshkol_madad2019`
- `age20_59_pcnt`
- `dist_to_cbd_km`
- `amenities_1km_area_wght` — 2019 commercial density (not PU detections)
- `share_local_poi_h_pre` — Jan 2020 share of named-POI hours inside the home radius

2023 WFH is **not** in X. Part 4b rematches inside SAs with vs without remote workers in 2023 (`high_wfh_2023`).

After matching, Panel A should show X balanced and third-place intensity still
different. Panel B reports SMD for pre-period `y`s (each metric × all-day / midday);
15 of them are balance checks, not extra matching covariates.

**Method:** standardise covariates → logit PS → 1:1 nearest-neighbour on logit PS (caliper = 0.1 × SD), without replacement.


In [ ]:
# Love plot — SMD before vs after PSM (1:1 matched sample)
# Panel A: PS covariates + intensities. Panel B: pre-period outcome shares.


ps_balance_vars = [
    ("eshkol_madad2019", "2019 SES cluster"),
    ("age20_59_pcnt", "2019 age 20–59 share"),
    ("dist_to_cbd_km", "Distance to CBD (km)"),
    ("amenities_1km_area_wght", "Amenity weight 1 km"),
    ("obs_h_per_day_pre", "Observed hours per day, pre"),
    ("share_local_poi_h_pre", "Jan 2020 local | all POIs, hours"),
    ("tp_intensity", "Detected TPs / 1,000 (not in PS)"),
    ("cand_intensity", "Candidates / 1,000 (not in PS)"),
    ("atm_intensity", "RW-atmosphere candidates / 1,000 (not in PS)"),
]

pre_share_vars = []
for col in SHARE_OUTCOME_COLS:
    lab = SHARE_OUTCOME_LABELS[col]
    in_ps = " (in PS)" if col == "share_local_poi_h" else " (not in PS)"
    pre_share_vars.append((f"{col}_pre", f"{lab}, all-day{in_ps}"))
    pre_share_vars.append((f"{col}_midday_pre", f"{lab}, midday (not in PS)"))


In [ ]:
psm = run_psm_did(
    lambda sa: sa["n_detected"] > 0,
    ps_balance_vars=ps_balance_vars,
    pre_share_vars=pre_share_vars,
)


In [ ]:
# Matched 1:1 sample: treated vs control (split from psm.matched)
flag = psm.flag_col
treated = psm.matched.loc[psm.matched[flag] == 1]
control = psm.matched.loc[psm.matched[flag] == 0]
num = psm.matched.select_dtypes(include="number")
skip = {"pair_id", flag}
cols = [c for c in num.columns if c not in skip]
print(f"treated n={len(treated):,}  control n={len(control):,}")
pd.concat(
    {"treated": treated[cols].mean(), "control": control[cols].mean()},
    axis=1,
)

### Part 4b — Split-sample DiD by 2023 neighbourhood WFH

**Goal.** Among home SAs that have remote workers in 2023, does third-place activation still shift local POI shares? Complementary sample: SAs below the WFH cut.

**Steps.**
1. `high_wfh_2023` = 1 if Jan–Mar 2023 SA-mean `flag_remote_work_smoothed_max_thresh` ≥ `WFH_HIGH_CUT` (0.2).
2. Rematch 1:1 **inside** each subsample (same X as Table 1; WFH is not in X).
3. TWFE Treat × Post in each matched subsample.

2023 WFH is measured in the post window, so this is a heterogeneity split, not a confounder control.


In [ ]:
# Rematch inside 2023 high- vs low-WFH SAs. Does not overwrite the main `psm` object.
PRIMARY_SPLIT_OUTCOME = "share_local_poi_h"


def run_psm_did_wfh_split(high, *, plot=False):
    flag = int(high)
    lab = (
        f"2023 high WFH (`{WFH_POST_COL}` ≥ {WFH_HIGH_CUT})"
        if flag == 1
        else f"2023 low WFH (`{WFH_POST_COL}` < {WFH_HIGH_CUT})"
    )
    sub = df_sa.loc[df_sa[WFH_HIGH_COL] == flag].copy()
    print(f"\n=== {lab} ===")
    print(f"SAs in split: {len(sub):,}  "
          f"(of {df_sa[WFH_HIGH_COL].notna().sum():,} with 2023 WFH)")
    if TREAT_FLAG in sub.columns:
        print(sub[TREAT_FLAG].value_counts().rename("n_SA"))
    if sub[TREAT_FLAG].nunique() < 2:
        print("Skip: treatment is constant in this subsample.")
        return None
    res = run_psm_did(
        sa=sub, plot=plot, verbose=True, set_globals=False,
        ps_balance_vars=ps_balance_vars, pre_share_vars=pre_share_vars,
    )
    return res


psm_wfh_high = run_psm_did_wfh_split(1)
psm_wfh_low = run_psm_did_wfh_split(0)

rows = []
for lab, res in [
    (f"high WFH (≥ {WFH_HIGH_CUT})", psm_wfh_high),
    (f"low WFH (< {WFH_HIGH_CUT})", psm_wfh_low),
]:
    if res is None or res.did is None or not len(res.did):
        continue
    n_pairs = int((res.matched[res.flag_col] == 1).sum()) if res.matched is not None else 0
    d = res.did[res.did["outcome"] == PRIMARY_SPLIT_OUTCOME].copy()
    d["split"] = lab
    d["n_pairs"] = n_pairs
    rows.append(d)

did_wfh_split = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if len(did_wfh_split):
    _show = [c for c in (
        "split", "win", "label", "coef", "se", "p", "sig",
        "ci_low", "ci_high", "n_pairs", "n_sa", "n_clusters",
    ) if c in did_wfh_split.columns]
    print("\n=== Treat × post on `share_local_poi_h` by 2023 WFH split ===")
    print(did_wfh_split[_show].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
    print(
        f"**Table 1c.** Split-sample TWFE DiD of treat × post on `{PRIMARY_SPLIT_OUTCOME}`, "
        f"after 1:1 PSM inside 2023 high- vs low-WFH home SAs. "
        f"*Notes:* `{WFH_HIGH_COL}` = 1{{{WFH_POST_COL} >= {WFH_HIGH_CUT}}}; "
        f"{WFH_POST_COL} is the Jan–Mar 2023 SA mean of "
        f"`flag_remote_work_smoothed_max_thresh`. Same X and caliper as Table 1 "
        f"(WFH is not in X). SA and month FE; SEs clustered on locality. "
        f"2023 WFH is post-treatment."
    )

    fig, ax = plt.subplots(figsize=(7.2, 3.8))
    wins = [w for w in DID_WINDOWS if w in set(did_wfh_split["win"])]
    colours = {
        f"high WFH (≥ {WFH_HIGH_CUT})": "#2b6cb0",
        f"low WFH (< {WFH_HIGH_CUT})": "#dd6b20",
    }
    for i, (split, sub) in enumerate(did_wfh_split.groupby("split", sort=False)):
        sub = sub.set_index("win").reindex(wins)
        x = np.arange(len(wins)) + (i - 0.5) * 0.18
        ax.errorbar(
            x, sub["coef"],
            yerr=[sub["coef"] - sub["ci_low"], sub["ci_high"] - sub["coef"]],
            fmt="o", ms=6, color=colours.get(split, "#4a5568"),
            ecolor=colours.get(split, "#4a5568"), elinewidth=1.4, capsize=3,
            label=split,
        )
    ax.axhline(0, ls="--", color="#718096", lw=1.0)
    ax.set_xticks(np.arange(len(wins)))
    ax.set_xticklabels(wins)
    ax.set_xlabel("Time-of-day window")
    ax.set_ylabel("DiD coefficient (share points)")
    ax.set_title("Treat × post on local POI hours, by 2023 neighbourhood WFH")
    ax.legend(fontsize=8, loc="best")
    plt.tight_layout()
    plt.show()
    print(
        f"**Fig. 2c.** Treat × post on `{PRIMARY_SPLIT_OUTCOME}` after 1:1 PSM "
        f"inside 2023 high-WFH vs low-WFH statistical areas, with 95% cluster-robust CIs. "
        f"*Notes:* High WFH = SA-mean flag ≥ {WFH_HIGH_CUT} in Jan–Mar 2023. "
        f"Each point is a separate rematch + TWFE DiD. Pre = {PRE_LABEL}; post = {POST_LABEL}."
    )
else:
    print("No split-sample DiD rows to plot.")


## Test: both groups start to spend more time locally than before

In [ ]:
# PSM 1:1 + Table 1 TWFE DiD (treat × post). Love plot / Fig. 2 print here.
# Default: run_psm_did() uses treat_rule from the config cell.
psm = run_psm_did(
    lambda sa: sa["tp_intensity"] >= 0.42,
    ps_balance_vars=ps_balance_vars,
    pre_share_vars=pre_share_vars,
)


In [ ]:
# Matching panel for CBGPS is built in Part 3 (`panel_m`).
# Binary 1:1 PSM is run_psm_did() in the Table 1 cell — do not rematch here.
print(f"Matching sample already on panel_m: {len(panel_m):,} SAs")
print(panel_m[TREAT_FLAG].value_counts().rename("n_SA"))


## Part 5 — Regression estimates (difference-in-differences)

**Goal.** Estimate how much weekday local-share outcomes shifted after third-place
activation, relative to matched controls, using the **same binary treatment as PSM**.

**Steps.**
1. `run_psm_did()` builds the SA × month × window panel on 1:1 matched pairs, absorbs SA and month FE, and clusters on locality. Edit `treat_rule` (or pass a callable) to rematch; DiD stays binary treat × post.
2. Table 1 and Fig. 1–2 are printed by that call (local shares × `all24` / `morning` / `midday` / `evening`).
3. Fig. 3: event study in the midday window (2019 lead vs Jan 2020 vs post months).
4. Table 3: pair-FE ANCOVA and fractional logit on the matched all-day cross-section.

The estimand is **treat × post** on the PSM-matched sample. `Treat_i` is the binary flag from `treat_rule` (`TREAT_FLAG`, default `has_third_place`), identical to the PSM grouping. It is not `tp_intensity`. Overlapping CIs are not a test of the difference;
the regression handles pairing and the common time shock directly.

**Main specification (TWFE DiD)** on the SA × month panel within each time window:

$$y_{it} = \alpha_i + \gamma_t + \delta\,(\mathrm{Treat}_i \times Post_t) + \varepsilon_{it}$$

`Treat_i` is time-invariant, so it is absorbed by SA fixed effects except through the
interaction. `δ` is the treated-minus-control change in the share (0.01 = 1 percentage
point). Standard errors are clustered on **locality** (`semel_yishuv`).

**How the fixed effects are handled.** They are *absorbed* (within transformation) rather than
dummy-coded. Absorbing leaves a design of a handful of columns and flags a regressor with
no within variation as unidentified. Inference uses \(t_{G-1}\) with \(G\) localities.

**Outcomes.** Local shares (visits and hours) among: all named POIs; POIs not within
250 m of work; commercial named POIs; PU candidates (mechanism); and two
unnamed-inclusive shares — **all dwells** (WFH diagnostic) and **not-home dwells**.
Every column in `SHARE_OUTCOME_COLS` is a treat × post DiD, in all four windows.

The window triple-difference is not run: `all24` and `midday` overlap, so they cannot be
stacked as mutually exclusive windows.


In [ ]:
# ---------------------------------------------------------------------------
# Table 1 — TWFE DiD: local shares × windows on the 1:1 PSM sample
# One-liner (next cell): run_psm_did()
# Optional:  run_psm_did(lambda sa: sa["n_candidates_atm"] >= 1)
# DiD is binary treat_rule × post. Intensity never enters the PS.
# ---------------------------------------------------------------------------

MAIN_OUTCOMES = [(c, SHARE_OUTCOME_LABELS[c]) for c in SHARE_OUTCOME_COLS]
OUT_LABELS = dict(MAIN_OUTCOMES)


In [ ]:
# Table 1 (already fit in Part 4). Re-run that cell to rematch / change treat_rule.
did = psm.did
_show = [c for c in (
    "label", "win", "coef", "se", "p", "sig",
    "pre_mean_treated", "pct_of_pre", "n_obs", "n_sa", "n_clusters",
) if c in did.columns]
print(did[_show].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))
print("\n=== DiD coefficient by window (treat × post) ===")
print(did.pivot(index="label", columns="win", values="coef").round(4).to_string())

## GPS AND TWFE DIFFERENCE-IN-DIFFERENCES

Continuous CBGPS on `tp_intensity` is not the main GPS path: the dosage is zero-inflated with a long right tail, so CBPS weights collapse. Use **multinomial IPW** on `tp_bin` instead.

**Steps.**
1. `tp_bin` ∈ {0, 1, 2} from Part 2 (0 = no detected TP; 1 = non-zero ≤ p75 of positives; 2 = high dose).
2. Multinomial logit P(`tp_bin` | z-scored `PS_COVARIATES`); ATT weights \(W_i = P(T=0\mid X_i)/P(T=k\mid X_i)\) for bin \(k\), and \(W_i=1\) for controls.
3. If any covariate still has |weighted SMD| > 0.1 vs control, print a warning.
4. WLS TWFE: `did_bin1 = 1{bin=1} × post` and `did_bin2 = 1{bin=2} × post`, weights = `ipw_weight`. Wald test \(H_0:\delta_1=\delta_2\).

One-liner: `run_mnipw_did()`. Optional continuous CBGPS remains in the next cell for diagnostics only.


In [ ]:
# Main GPS path: multinomial IPW on tp_bin + WLS DiD (bin1 × post, bin2 × post).
# Re-run the MN-IPW function cell if this errors. No BQ.
# Optional diagnostics (continuous CBGPS failed on tp_intensity):
#   run_cbgps_did("share_tp")
mn = run_mnipw_did()


In [ ]:
mn_high = run_mnipw_did(
    panel=panel_m.loc[panel_m[WFH_HIGH_COL] == 1],
    set_globals=False,
)

In [ ]:
# Optional: share_tp is also built inside run_cbgps_did if missing.
panel_m = panel_m.copy()
cand = panel_m["cand_intensity"].astype(float)
panel_m["share_tp"] = np.where(
    cand > 0, panel_m["tp_intensity"].astype(float) / cand, 0.0
)

In [ ]:
mn_low = run_mnipw_did(
    panel=panel_m.loc[panel_m[WFH_HIGH_COL] == 0],
    set_globals=False,
)

## Part 7 — Robustness (continuous intensity / CBGPS)

Stress-tests of Table 1b (`INTENSITY_COL` × post on the eight local shares):

1. **Main** — CBGPS-weighted TWFE (already in `did_gps`).
2. **Worker-day × GPS weights** — `gps_w * n_worker_days` so thin SAs cannot dominate.
3. **Unmatched, no GPS** — same intensity DiD on `panel_m` without CBGPS weights.

Alternative dosages (`cand_intensity`, `atm_intensity`) are **not** in this table: they
need their own CBGPS fit. Run `run_cbgps_did("cand_intensity")` (or `"atm_intensity"`)
instead of swapping the dosage onto `tp_intensity` weights.


## Notes for the paper

- Main continuous specification: `run_cbgps_did("tp_intensity")` — CBGPS weights on covariate-complete SAs (`panel_m`), then TWFE of intensity × post on the eight local-share outcomes (`all24` and `midday`).
- GPS is fit on `log1p(intensity)` with `PS_COVARIATES`; DiD uses the original intensity scale. Intensities never enter the GPS covariate list.
- Alternative dosages (`cand_intensity`, `atm_intensity`) each need their own CBGPS fit: `run_cbgps_did("cand_intensity")`. Do not reuse `tp_intensity` weights.
- Placebo (Table 1c): pooled 2019 → Jan 2020 with the same weights — intensity × post should be insignificant before activation.
- Event study (Fig. 3): `INTENSITY_COL` × month, CBGPS-weighted, Jan 2020 = 0; 2019 is the lead.
- Robustness (Part 7): main CBGPS; `gps_w × n_worker_days`; unmatched intensity DiD (no GPS).
- Binary 1:1 PSM (Part 4 / Table 1 / Fig. 2) remains a separate specification; it is not required for the continuous path.
- Spatial radius re-estimation and odds / home–outbound composition are deferred (outcomes and helpers no longer match that design).
- `wfh_avg_post` is the Jan–Mar 2023 SA mean of `flag_remote_work_smoothed_max_thresh`. It is **not** a matching covariate. Part 4b rematches 1:1 inside `high_wfh_2023` vs the complement, then Treat × Post in each subsample. 2023 WFH is post-treatment; read the split as heterogeneity, not a causal moderator.
